## regression problem - salary prediction of MBA placement dataset  
### with feature selection, standardization, data preprocessing and deployment 

In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle

In [77]:
# feature selection
def selectkbest(indep_x,dep_y,n):
    test=SelectKBest(score_func=f_regression,k=n)
    fit1=test.fit(indep_x,dep_y)    # fit - learns which features are important features 
    seleted_features=fit1.transform(indep_x)     #transform - select those features 
    # seleted_features = test.fit_transform(indep_x,dep_y)        # fit & transform in one step 
    return seleted_features, fit1

# tran test split and standardization 
def split_scaler(indep_x,dep_y):
    x_train,x_test,y_train,y_test = train_test_split(indep_x,dep_y,test_size=0.20,random_state=0)
    # standardization - Input X
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    pickle.dump(sc,open("x_mba_scaler","wb"))   # saving the X i/p standardization in a file 
    return  x_train,x_test,y_train,y_test

# regression model predict & evaluation 
def r2_score_pred(regressor,x_test,y_test):
    y_pred=regressor.predict(x_test)
    from sklearn.metrics import r2_score
    r2_score=r2_score(y_test,y_pred)
    return r2_score 

# regression algorithms 
# linear
def linear(x_train,y_train,x_test):
    from sklearn.linear_model import LinearRegression
    regressor=LinearRegression()
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 
    
# SVR -linear
def svrl(x_train,y_train,x_test):
    from sklearn.svm import SVR
    regressor=SVR(kernel='linear')
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 
    
# SVR-non linear
def svrnl(x_train,y_train,x_test):
    from sklearn.svm import SVR
    regressor=SVR(kernel='rbf')
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 
    
# decision tree
def decisionReg(x_train,y_train,x_test):
    from sklearn.tree import DecisionTreeRegressor
    regressor=DecisionTreeRegressor(criterion='squared_error',random_state=0)
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 
    
# random forest 
def randomForestReg(x_train,y_train,x_test):
    from sklearn.ensemble import RandomForestRegressor
    regressor=RandomForestRegressor(n_estimators=10,random_state=0)
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 


# select features table 
def select_k_regression(acclinear,accsvrl,accsvrnl,accdes,accrf):
    dataframe=pd.DataFrame(index=['r_regression'],columns=['linear','svrl','svrnl','decisionReg','randomForestReg'])
    for number,idex in enumerate(dataframe.index):
        dataframe.loc[idex,'linear']=acclinear[number]
        dataframe.loc[idex,'svrl']=accsvrl[number]
        dataframe.loc[idex,'svrnl']=accsvrnl[number]
        dataframe.loc[idex,'decisionReg']=accdes[number]
        dataframe.loc[idex,'randomForestReg']=accrf[number]
        return dataframe

In [78]:
data_original = pd.read_csv('preproc_placement.csv')
data_mba= data_original
data_mba

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,0.0
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,211,M,80.60,Others,82.00,Others,Commerce,77.60,Comm&Mgmt,No,91.0,Mkt&Fin,74.49,Placed,400000.0
211,212,M,58.00,Others,60.00,Others,Science,72.00,Sci&Tech,No,74.0,Mkt&Fin,53.62,Placed,275000.0
212,213,M,67.00,Others,67.00,Others,Commerce,73.00,Comm&Mgmt,Yes,59.0,Mkt&Fin,69.72,Placed,295000.0
213,214,F,74.00,Others,66.00,Others,Commerce,58.00,Comm&Mgmt,No,70.0,Mkt&HR,60.23,Placed,204000.0


In [79]:
data_mba=pd.get_dummies(data_mba,drop_first=True,dtype=int)
data_mba.head()

,sl_no,ssc_p,hsc_p,degree_p,etest_p,mba_p,salary,gender_M,ssc_b_Others,hsc_b_Others,hsc_s_Commerce,hsc_s_Science,degree_t_Others,degree_t_Sci&Tech,workex_Yes,specialisation_Mkt&HR,status_Placed
0,1,67.00,91.00,58.00,55.0,58.80,270000.0,1,1,1,1,0,0,1,0,1,1
1,2,79.33,78.33,77.48,86.5,66.28,200000.0,1,0,1,0,1,0,1,1,0,1
2,3,65.00,68.00,64.00,75.0,57.80,250000.0,1,0,0,0,0,0,0,0,0,1
3,4,56.00,52.00,52.00,66.0,59.43,0.0,1,0,0,0,1,0,1,0,1,0
4,5,85.80,73.60,73.30,96.8,55.50,425000.0,1,0,0,1,0,0,0,0,0,1


In [80]:
# ip op split 
indep_x=data_mba.drop('salary',axis=1)
dep_y = data_mba['salary']

In [81]:
indep_x.head()

,sl_no,ssc_p,hsc_p,degree_p,etest_p,mba_p,gender_M,ssc_b_Others,hsc_b_Others,hsc_s_Commerce,hsc_s_Science,degree_t_Others,degree_t_Sci&Tech,workex_Yes,specialisation_Mkt&HR,status_Placed
0,1,67.00,91.00,58.00,55.0,58.80,1,1,1,1,0,0,1,0,1,1
1,2,79.33,78.33,77.48,86.5,66.28,1,0,1,0,1,0,1,1,0,1
2,3,65.00,68.00,64.00,75.0,57.80,1,0,0,0,0,0,0,0,0,1
3,4,56.00,52.00,52.00,66.0,59.43,1,0,0,0,1,0,1,0,1,0
4,5,85.80,73.60,73.30,96.8,55.50,1,0,0,1,0,0,0,0,0,1


In [82]:
dep_y.head()

0    270000.0
1    200000.0
2    250000.0
3         0.0
4    425000.0
Name: salary, dtype: float64

In [83]:
# k best 
no_of_features = 4
kBest, fit1=selectkbest(indep_x,dep_y,no_of_features)
kBest

array([[67.  , 91.  , 58.  ,  1.  ],
       [79.33, 78.33, 77.48,  1.  ],
       [65.  , 68.  , 64.  ,  1.  ],
       [56.  , 52.  , 52.  ,  0.  ],
       [85.8 , 73.6 , 73.3 ,  1.  ],
       [55.  , 49.8 , 67.25,  0.  ],
       [46.  , 49.2 , 79.  ,  0.  ],
       [82.  , 64.  , 66.  ,  1.  ],
       [73.  , 79.  , 72.  ,  1.  ],
       [58.  , 70.  , 61.  ,  0.  ],
       [58.  , 61.  , 60.  ,  1.  ],
       [69.6 , 68.4 , 78.3 ,  1.  ],
       [47.  , 55.  , 65.  ,  0.  ],
       [77.  , 87.  , 59.  ,  1.  ],
       [62.  , 47.  , 50.  ,  0.  ],
       [65.  , 75.  , 69.  ,  1.  ],
       [63.  , 66.2 , 65.6 ,  1.  ],
       [55.  , 67.  , 64.  ,  0.  ],
       [63.  , 66.  , 64.  ,  0.  ],
       [60.  , 67.  , 70.  ,  1.  ],
       [62.  , 65.  , 66.  ,  1.  ],
       [79.  , 76.  , 85.  ,  1.  ],
       [69.8 , 60.8 , 72.23,  1.  ],
       [77.4 , 60.  , 64.74,  1.  ],
       [76.5 , 97.7 , 78.86,  1.  ],
       [52.58, 54.6 , 50.2 ,  0.  ],
       [71.  , 79.  , 66.  ,  1.  ],
 

In [84]:
indep_x.columns[fit1.get_support()]

Index(['ssc_p', 'hsc_p', 'degree_p', 'status_Placed'], dtype='object')

In [85]:
acclinear=[]
accsvrl=[]
accsvrnl=[]
accdes=[]
accrf=[]

x_train,x_test,y_train,y_test=split_scaler(kBest,dep_y)

# algorithms 
r2=linear(x_train,y_train,x_test)
acclinear.append(r2)
r2=svrl(x_train,y_train,x_test)
accsvrl.append(r2)
r2=svrnl(x_train,y_train,x_test)
accsvrnl.append(r2)
r2=decisionReg(x_train,y_train,x_test)
accdes.append(r2)
r2=randomForestReg(x_train,y_train,x_test)
accrf.append(r2)

result_regression = select_k_regression(acclinear,accsvrl,accsvrnl,accdes,accrf)

In [86]:
result_regression #4

,linear,svrl,svrnl,decisionReg,randomForestReg
r_regression,0.899295,-0.320522,-0.324558,-0.098962,0.87083


In [87]:
def linear(x_train,y_train,x_test):
    from sklearn.linear_model import LinearRegression
    regressor=LinearRegression()
    regressor.fit(x_train,y_train)
    r2 = r2_score_pred(regressor,x_test,y_test)
    return r2 

final_model =linear(x_train,y_train,x_test)
final_model

0.8992950766841188

# lINEAR REGRESSION 
## Machine Learning Pipeline with Automated Preprocessing + Model Selection

## linear model creation 

In [90]:
data_original.head()

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,0.0
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0


In [91]:
data_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    object 
 2   ssc_p           215 non-null    float64
 3   ssc_b           215 non-null    object 
 4   hsc_p           215 non-null    float64
 5   hsc_b           215 non-null    object 
 6   hsc_s           215 non-null    object 
 7   degree_p        215 non-null    float64
 8   degree_t        215 non-null    object 
 9   workex          215 non-null    object 
 10  etest_p         215 non-null    float64
 11  specialisation  215 non-null    object 
 12  mba_p           215 non-null    float64
 13  status          215 non-null    object 
 14  salary          215 non-null    float64
dtypes: float64(6), int64(1), object(8)
memory usage: 25.3+ KB


In [92]:
# input output split 
X=data_original.drop(['salary','sl_no'], axis=1)
Y=data_original['salary']

In [93]:
# train test split 
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.30,random_state=42)

In [94]:
# column identification
categorical_cols = X.select_dtypes(include=['object']).columns
print(categorical_cols)
numerical_cols = X.select_dtypes(exclude=['object']).columns
print(numerical_cols)

Index(['gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex',
       'specialisation', 'status'],
      dtype='object')
Index(['ssc_p', 'hsc_p', 'degree_p', 'etest_p', 'mba_p'], dtype='object')


In [100]:
# preprocessing  -  for input
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
preprocessor = ColumnTransformer(
    transformers=[('cat',OneHotEncoder(drop='first',handle_unknown='ignore'), categorical_cols ),
                 ('num',"passthrough" ,numerical_cols )
    ]
)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [103]:
# Pipeline --> feature selection + model creation 
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
pipeline = Pipeline(
    [
        ("preprocessing",preprocessor),
        ("scaling",StandardScaler()),
        ("feature_selection",SelectKBest(score_func=f_regression,k=4)),
        ("model",LinearRegression())
    ]
)
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('scaling', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfo

In [104]:

# model creation
pipeline.fit(x_train,y_train)
y_predicted = pipeline.predict(x_test)

# model evaluation 
from sklearn.metrics import r2_score
r_score = r2_score(y_test,y_predicted)
print("regression accuracy: ", r_score)

regression accuracy:  0.7839540966215024


In [109]:
# gender	ssc_p	ssc_b	hsc_p	hsc_b	hsc_s	degree_p	degree_t	workex	etest_p	specialisation	mba_p	status	salary
# M	67.00	Others	91.00	Others	Commerce	58.00	Sci&Tech	No	55.0	Mkt&HR	58.80	Placed	270000.0

# Predict with raw categorical input
user_input = pd.DataFrame({"gender":['M'],
                           "ssc_p":[67.00],
                           "ssc_b":['Others'],
                           "hsc_p":[91.00],
                           "hsc_b":['Others'],
                           "hsc_s":['Commerce'],
                           "degree_p":[58.00],
                           "degree_t":['Sci&Tech'],
                           "workex":['No'],
                           "etest_p":[55.0],
                           "specialisation":['Mkt&HR'],
                           "mba_p":[58.80],
                           "status":['Placed']
                            })
print(pipeline.predict(user_input))

[309802.16134968]
